# Trích Xuất Metadata File Âm Thanh

**Mục tiêu**: Đọc các file `.wav` đã được normalize trong thư mục `data/processed/`, trích xuất metadata tương ứng và lưu trữ kết quả để tra cứu. Kết quả trả về được lưu thành file `.csv`.

Metadata gồm có 2 loại chính:
1. **Non-acoustic Metadata** (không liên quan trực tiếp đến sóng âm): Tên định danh file, độ dài sau khi cắt gọt, số từ khóa (keyword) được nói ra (nếu có).
2. **Acoustic Metadata** (liên quan đến sóng âm / lý học): Tốc độ lấy mẫu, mức độ to nhỏ trung bình (RMS) dB, và độ phân giải dữ liệu.

### Định Nghĩa Các Trường Dữ Liệu

| Trường | Nhóm | Giải thích | Ví Dụ |
|---|---|---|---|
| `speaker` | Non-acoustic | ID người nói (chính là tên thư mục parent) | `p225` |
| `file_name` | Non-acoustic | Tên file vật lí | `p225_001.wav` |
| `duration_sec`| Non-acoustic | Chiều dài audio thu được (tính bằng giây) | `5.0` |
| `keyword` | Non-acoustic | Nội dung transcript trích từ dataset gốc `.txt` | `"Please call Stella."` |
| `sample_rate` | Acoustic | Tần số mẫu (Hz) sau resampling. | `16000` |
| `rms_db` | Acoustic | Năng lượng RMS trung bình, quy đổi hệ dBFS | `-26.34` |
| `bit_depth` | Acoustic | Độ sâu / phân giải của mẫu PCM | `16` |

---

## 1. Import Thư Viện và Khai Báo Đường Dẫn

In [1]:
import os
import numpy as np
import librosa
import soundfile as sf
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [2]:
# Đường dẫn thư mục
PROJECT_ROOT  = Path().resolve().parents[1]  # src/extract_metadata lên 2 bậc để vào Beta/

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"    # Chứa file wav (Input)
METADATA_DIR  = PROJECT_ROOT / "data" / "metadata"     # Output thư mục 
TXT_DIR       = PROJECT_ROOT / "data" / "raw" / "VCTK-Corpus" / "VCTK-Corpus" / "txt"  # Source chứa file text keyword

METADATA_CSV_PATH = METADATA_DIR / "metadata.csv"

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"TXT_DIR       : {TXT_DIR}")
print(f"Output        : {METADATA_CSV_PATH}")

PROJECT_ROOT  : C:\BTL_CSDLDPT\Beta
PROCESSED_DIR : C:\BTL_CSDLDPT\Beta\data\processed
TXT_DIR       : C:\BTL_CSDLDPT\Beta\data\raw\VCTK-Corpus\VCTK-Corpus\txt
Output        : C:\BTL_CSDLDPT\Beta\data\metadata\metadata.csv


## 2. Hàm Xử Lý Thành Phần

Chúng ta cần hai component xử lý chính:
1. Lấy dòng transcript (keyword) từ text folder của VCTK dựa trên tên file.
2. Trích xuất đặc trưng vật lý của audio sử dụng thư viện `librosa` và `soundfile`.

In [3]:
def lookup_keyword(speaker: str, file_name_without_ext: str) -> str:
    """
    Truy lục đoạn hội thoại trong dataset gốc của VCTK
    
    VD:
        speaker = p225
        file_name_without_ext = p225_001
        --> Sẽ trỏ tới file data/raw/VCTK-Corpus/VCTK-Corpus/txt/p225/p225_001.txt
    """
    txt_file_path = TXT_DIR / speaker / f"{file_name_without_ext}.txt"
    keyword = ""
    if txt_file_path.exists():
        try:
            with open(txt_file_path, 'r', encoding='utf-8') as f:
                keyword = f.read().strip()
        except Exception as e:
            log.warning(f"Lỗi đọc text file {txt_file_path.name}: {e}")
    return keyword

In [4]:
def extract_audio_metadata(wav_path: Path) -> dict:
    """
    Thu thập Non-acoustic Metadata và Acoustic Metadata từ một file .wav.
    """
    speaker = wav_path.parent.name
    file_name = wav_path.name
    
    # 1. Đọc metadata thuần tuý (rất nhanh, không cần load toàn bộ mảng sóng vào ram)
    info = sf.info(str(wav_path))
    
    # 2. Xử lí Bit depth (soundfile subtype)
    # VCTK là PCM_16 nhưng tuỳ vào quá trình preprocessing, có thể soundfile đã transform type
    bit_depth = {
        "PCM_16": 16, 
        "PCM_24": 24, 
        "PCM_32": 32, 
        "FLOAT": 32
    }.get(info.subtype, 16)

    # 3. Phân tích Acoustic (Load file)
    # sr=None: Giữ nguyên sample rate bên trong file (đã ở mức 16000)
    y, sr = librosa.load(str(wav_path), sr=None, mono=True)
    
    # RMS calculation & convert to decibel (dBFS system)
    # Cộng 1e-9 nhỏ xíu để không log10 bị -vô cực với file nhiễu bằng 0.
    rms_linear = float(np.sqrt(np.mean(y ** 2)))
    rms_db     = float(20 * np.log10(rms_linear + 1e-9))
    
    # Actual length
    duration_sec = float(len(y) / sr)
    
    # 4. Tra cứu Keyword
    keyword = lookup_keyword(speaker, wav_path.stem)

    return {
        "speaker"      : speaker,
        "file_name"    : file_name,
        "duration_sec" : round(duration_sec, 3),
        "keyword"      : keyword,
        "sample_rate"  : int(sr),
        "rms_db"       : round(rms_db, 4),
        "bit_depth"    : bit_depth
    }

## 3. Quét Toàn Bộ File Audio Và Lưu Xuống CSV

In [5]:
wav_files = sorted(PROCESSED_DIR.rglob("*.wav"))
log.info(f"Đã tìm thấy {len(wav_files)} file .wav\n")

metadata_records = []

for wav_path in tqdm(wav_files, desc="Trích xuất Metadata"):
    try:
        record = extract_audio_metadata(wav_path)
        metadata_records.append(record)
    except Exception as e:
        log.error(f"Lỗi đọc file {wav_path}: {e}")

20:47:08 [INFO] Đã tìm thấy 1012 file .wav



Trích xuất Metadata:   0%|          | 0/1012 [00:00<?, ?it/s]

In [6]:
# Tạo thư mục output nếu chưa tồn tại
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# Chuyển đối JSON objects thành array 2D của Pandas rồi xuất
df = pd.DataFrame(metadata_records)
df.to_csv(METADATA_CSV_PATH, index=False, encoding='utf-8')

log.info(f"Hoàn tất! Cấu trúc lưu trên disk:\n {METADATA_CSV_PATH}")

20:47:55 [INFO] Hoàn tất! Cấu trúc lưu trên disk:
 C:\BTL_CSDLDPT\Beta\data\metadata\metadata.csv


## 4. Kiểm Tra Thống Kê

In [7]:
print(df.info())

print("\n\nThống kê Acoustic Metadata:\n")
print(df[["duration_sec", "sample_rate", "rms_db"]].describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1012 entries, 0 to 1011
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   speaker       1012 non-null   object 
 1   file_name     1012 non-null   object 
 2   duration_sec  1012 non-null   float64
 3   keyword       1012 non-null   object 
 4   sample_rate   1012 non-null   int64  
 5   rms_db        1012 non-null   float64
 6   bit_depth     1012 non-null   int64  
dtypes: float64(2), int64(2), object(3)
memory usage: 55.5+ KB
None


Thống kê Acoustic Metadata:

       duration_sec  sample_rate       rms_db
count        1012.0       1012.0  1012.000000
mean            5.0      16000.0   -28.552915
std             0.0          0.0     3.268052
min             5.0      16000.0   -50.713500
25%             5.0      16000.0   -30.789775
50%             5.0      16000.0   -28.365650
75%             5.0      16000.0   -26.250725
max             5.0      16000.0   -18.9

In [8]:
df.head()

,speaker,file_name,duration_sec,keyword,sample_rate,rms_db,bit_depth
0,p225,p225_010.wav,5.0,"People look, but no one ever finds it.",16000,-20.8670,16
1,p225,p225_033.wav,5.0,"It's really no great surprise, because the pri...",16000,-22.7320,16
2,p225,p225_056.wav,5.0,"Clearly, it can go either way.",16000,-24.7270,16
3,p225,p225_057.wav,5.0,"However, the City was in no doubt about the ul...",16000,-20.5408,16
4,p225,p225_120.wav,5.0,The industry is not well organised.,16000,-26.7687,16
